# Export to Volume

Exports MLflow models from the source catalog to the export volume. Writes manifest and metadata per version. Model names in target are the same as source (no suffix).

## Configuration

Read widget parameters: source/target catalogs, schema, volume name, and the comma-separated list of model names to export.

In [ ]:
dbutils.widgets.text("source_catalog", "", "Source catalog")
dbutils.widgets.text("target_catalog", "", "Target catalog (for manifest)")
dbutils.widgets.text("source_schema", "default", "Source schema")
dbutils.widgets.text("target_schema", "default", "Target schema (for manifest)")
dbutils.widgets.text("volume_name", "model_exports", "Export volume name")
dbutils.widgets.text("model_names", "", "Model names comma-separated")

source_catalog = dbutils.widgets.get("source_catalog").strip()
target_catalog = dbutils.widgets.get("target_catalog").strip()
source_schema = dbutils.widgets.get("source_schema").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
volume_name = dbutils.widgets.get("volume_name").strip()
model_list = [m.strip() for m in dbutils.widgets.get("model_names").strip().split(",") if m.strip()]

if not source_catalog or not target_catalog or not model_list:
    raise ValueError("Set source_catalog, target_catalog, and model_names")

MODELS = [(f"{source_catalog}.{source_schema}.{m}", f"{target_catalog}.{target_schema}.{m}") for m in model_list]
VOLUME_ROOT = f"/Volumes/{source_catalog}/{source_schema}/{volume_name}"
print(f"Volume: {VOLUME_ROOT}")
for src, tgt in MODELS:
    print(f"  {src} -> {tgt}")

## Check for READY model versions

Counts READY versions across all models. Exits early if nothing to export.

In [ ]:
import mlflow
from mlflow import MlflowClient
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
total_ready = sum(
    sum(1 for v in client.search_model_versions(filter_string=f"name='{s}'") if getattr(v, "status", "READY") == "READY")
    for s, _ in MODELS
)
if total_ready == 0:
    print("No READY versions; skipping.")
    dbutils.notebook.exit("SKIP_NO_MODELS")
print(f"Found {total_ready} READY version(s); proceeding.")

## Prepare export volume

Ensures the export volume exists and clears any previous content.

In [ ]:
import os, shutil, json
from datetime import datetime

def safe_aliases(mv):
    try:
        a = mv.aliases
        return list(a()) if callable(a) else list(a or [])
    except Exception:
        return []

spark.sql(f"CREATE VOLUME IF NOT EXISTS {source_catalog}.{source_schema}.{volume_name}")
os.makedirs(VOLUME_ROOT, exist_ok=True)
for entry in os.listdir(VOLUME_ROOT):
    full = os.path.join(VOLUME_ROOT, entry)
    if os.path.isdir(full):
        shutil.rmtree(full)
    else:
        os.remove(full)
print(f"Volume ready: {VOLUME_ROOT}")

## Export models and write manifest

For each model, downloads artifacts and metadata (metrics, params, tags, aliases, signature) per version, then writes a global manifest summarizing all exported models.

In [ ]:
global_results = []
for source_model, target_model in MODELS:
    model_dir_name = source_model.replace(".", "__")
    model_root = f"{VOLUME_ROOT}/{model_dir_name}"
    os.makedirs(model_root, exist_ok=True)
    ready = sorted([v for v in client.search_model_versions(filter_string=f"name='{source_model}'") if getattr(v, 'status', 'READY') == 'READY'], key=lambda v: int(v.version))
    ver_to_aliases = {}
    try:
        rm = client.get_registered_model(source_model)
        al = getattr(rm, "aliases", None) or []
        if isinstance(al, dict):
            for k, v in al.items(): ver_to_aliases.setdefault(str(v), []).append(str(k))
        else:
            for a in al:
                if hasattr(a, "alias"): ver_to_aliases.setdefault(str(a.version), []).append(a.alias)
                elif isinstance(a, dict): ver_to_aliases.setdefault(str(a["version"]), []).append(a["alias"])
    except Exception as e:
        print(f"  WARN: could not list aliases for {source_model}: {type(e).__name__}: {e}")
    if ver_to_aliases:
        print(f"  Aliases on source: {ver_to_aliases}")
    exported = []
    for mv in ready:
        ver, vdir = mv.version, f"{model_root}/v{mv.version}"
        os.makedirs(vdir, exist_ok=True)
        model_uri = f"models:/{source_model}/{ver}"
        import tempfile
        with tempfile.TemporaryDirectory() as tmp:
            local_path = mlflow.artifacts.download_artifacts(artifact_uri=model_uri, dst_path=tmp)
            model_dst = f"{vdir}/model"
            if os.path.exists(model_dst):
                shutil.rmtree(model_dst)
            shutil.copytree(local_path, model_dst)
        try:
            run = client.get_run(mv.run_id)
            metrics, params, tags = dict(run.data.metrics), dict(run.data.params), dict(run.data.tags)
        except Exception:
            metrics, params, tags = {}, {}, {}
        sig_dict = None
        try:
            mi = mlflow.models.get_model_info(model_uri)
            if mi.signature:
                sig_dict = {"inputs": mi.signature.inputs.to_dict() if mi.signature.inputs else None, "outputs": mi.signature.outputs.to_dict() if mi.signature.outputs else None}
        except Exception:
            pass
        if sig_dict is None and os.path.isdir(model_dst):
            try:
                mi_local = mlflow.models.get_model_info(f"file://{os.path.abspath(model_dst)}")
                if getattr(mi_local, "signature", None):
                    sig_dict = {"inputs": mi_local.signature.inputs.to_dict(), "outputs": mi_local.signature.outputs.to_dict() if mi_local.signature.outputs else None}
            except Exception:
                pass
        meta = {"source_model": source_model, "target_model": target_model, "source_version": str(ver), "source_run_id": mv.run_id, "metrics": metrics, "params": params, "tags": tags, "aliases": ver_to_aliases.get(str(ver), []), "signature": sig_dict, "exported_at": datetime.now().isoformat()}
        with open(f"{vdir}/metadata.json", "w") as f:
            json.dump(meta, f, indent=2, default=str)
        exported.append(meta)
    with open(f"{model_root}/model_manifest.json", "w") as f:
        json.dump({"source_model": source_model, "target_model": target_model, "versions": len(exported), "exported_at": datetime.now().isoformat()}, f, indent=2)
    global_results.append((source_model, target_model, len(exported)))

with open(f"{VOLUME_ROOT}/manifest.json", "w") as f:
    json.dump({"models": [{"source": s, "target": t, "dir": s.replace(".", "__"), "versions": n} for s, t, n in global_results], "total_models": len(global_results), "total_versions": sum(n for _, _, n in global_results), "exported_at": datetime.now().isoformat()}, f, indent=2)
print("EXPORT SUMMARY")
for s, t, n in global_results:
    print(f"  {s} -> {t}  ({n} versions)")

## Export UC permissions (grants) per model

Reads direct grants on each source registered model via the UC permissions REST endpoint and writes them to `grants.json` under that model's export folder. The import step replays them on the target model. Inherited grants (from catalog/schema) are not exported -- only direct grants on the model itself.

In [ ]:
from databricks.sdk import WorkspaceClient
_w = WorkspaceClient()
total_grants = 0
for source_model, target_model in MODELS:
    model_dir_name = source_model.replace(".", "__")
    model_root = f"{VOLUME_ROOT}/{model_dir_name}"
    grants_payload = {"source_model": source_model, "target_model": target_model, "privilege_assignments": []}
    try:
        resp = _w.api_client.do("GET", f"/api/2.1/unity-catalog/permissions/function/{source_model}")
        grants_payload["privilege_assignments"] = resp.get("privilege_assignments", []) if isinstance(resp, dict) else []
    except Exception as e:
        print(f"  WARN: could not read grants for {source_model}: {type(e).__name__}: {e}")
    with open(f"{model_root}/grants.json", "w") as f:
        json.dump(grants_payload, f, indent=2)
    n = len(grants_payload["privilege_assignments"])
    total_grants += n
    print(f"  {source_model}: {n} grant(s) exported")
    for pa in grants_payload["privilege_assignments"]:
        print(f"    - {pa.get('principal')}: {pa.get('privileges')}")
print(f"GRANTS EXPORT: {total_grants} privilege_assignment(s) across {len(MODELS)} model(s)")